# CNN training file
This file contains training code for the first model, that predicts one out of the 5 classes, stores, homes, public spaces, leisure and working spaces. This is nearly identical to the first part of the CNNN feature extractor file that we had   

This file also contains an extra model to predict from one of the 5 categories into a room type.  

In [ ]:
# for jupyter notebook
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import os

# train test split
from sklearn.model_selection import train_test_split

# tensorflow
import tensorflow as tf
import keras

# keras stuff
from keras.datasets import mnist
from keras.models import Model
from keras.layers import Dense, Input, Conv2D, MaxPooling2D, Flatten, Dropout
from keras.optimizers import Adam, SGD
from keras.losses import sparse_categorical_crossentropy, binary_crossentropy

# model selection and stucture
1 lightweight model to place an indoor image to one of the following categories: working_spacecs, store, public_spaces, leisure, home  

Then we will have 5 models for each of these 5 categories to assign them to even more specific categories.  

The training will be done only on a fraction of the images for now for the sake of reducing training times  

In [ ]:
# for Le Fan's google drive
working_dir = 'drive/MyDrive/DS4420 project/'
images_dir = 'drive/MyDrive/DS4420 project/Images'
print(os.listdir(images_dir))
# check GPU
tf.config.list_physical_devices('GPU')

['store', 'home', 'public_spaces', 'leisure', 'working_spaces']


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [ ]:
# # preprocessing file will change the dataset under /Images into 5 categories from the MIT dataset, use this if running in local files, not jupyter notebook
# !python data_preprocessing.py

In [ ]:
def folder_to_numpy(folder_path, target_size=(128, 128)):
    images = []

    for i, subfolder in enumerate(os.listdir(folder_path)):
      print(f'Subfolder {i+1}/{len(os.listdir(folder_path))}')
      subfolder_path = os.path.join(folder_path, subfolder)

      for fname in os.listdir(subfolder_path):
        fpath = os.path.join(subfolder_path, fname)

        with Image.open(fpath) as img:
                img = img.convert('L')
                img = ImageOps.pad(img, target_size, color=0)
                images.append(np.array(img, dtype=np.uint8))

    return np.array(images, dtype=np.uint8)

In [ ]:
scene_folders = os.listdir(images_dir)
all_data = []
all_labels = []

for label, folder_name in enumerate(scene_folders):
  print(f'Folder {label+1}/{len(scene_folders)}')
  folder_path = os.path.join(images_dir, folder_name)
  data = folder_to_numpy(folder_path)
  labels = np.full(data.shape[0], label)
  all_data.append(data)
  all_labels.append(labels)

indoor_data = np.concatenate(all_data, axis=0)
y = np.concatenate(all_labels, axis=0)

Folder 1/5
Subfolder 1/12
Subfolder 2/12
Subfolder 3/12
Subfolder 4/12
Subfolder 5/12
Subfolder 6/12
Subfolder 7/12
Subfolder 8/12
Subfolder 9/12
Subfolder 10/12
Subfolder 11/12
Subfolder 12/12
Folder 2/5
Subfolder 1/14
Subfolder 2/14
Subfolder 3/14
Subfolder 4/14
Subfolder 5/14
Subfolder 6/14
Subfolder 7/14
Subfolder 8/14
Subfolder 9/14
Subfolder 10/14
Subfolder 11/14
Subfolder 12/14
Subfolder 13/14
Subfolder 14/14
Folder 3/5
Subfolder 1/15
Subfolder 2/15
Subfolder 3/15
Subfolder 4/15
Subfolder 5/15
Subfolder 6/15
Subfolder 7/15
Subfolder 8/15
Subfolder 9/15
Subfolder 10/15
Subfolder 11/15
Subfolder 12/15
Subfolder 13/15
Subfolder 14/15
Subfolder 15/15
Folder 4/5
Subfolder 1/11
Subfolder 2/11
Subfolder 3/11
Subfolder 4/11
Subfolder 5/11
Subfolder 6/11
Subfolder 7/11
Subfolder 8/11
Subfolder 9/11
Subfolder 10/11
Subfolder 11/11
Folder 5/5
Subfolder 1/15
Subfolder 2/15
Subfolder 3/15
Subfolder 4/15
Subfolder 5/15
Subfolder 6/15
Subfolder 7/15
Subfolder 8/15
Subfolder 9/15
Subfolder 10/1

For our initial model, we took in an image and classified it to one out of 67 classes, now we want to classify it into one out of 5 broader classes

In [ ]:
# split the data into training and test
x_train, x_test, y_train, y_test = train_test_split(indoor_data, y, test_size=0.2, random_state=1, stratify=y)

# Normalize image data
x_train = x_train / 255.0
x_test = x_test / 255.0

# Reshape to add channel dimension for Keras
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

In [ ]:
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import BatchNormalization

inpx = Input(shape=(128, 128, 1))

con_layer = Conv2D(32, kernel_size=(3,3), activation='relu', padding='same')(inpx)
con_layer = BatchNormalization()(con_layer)
pool_layer = MaxPooling2D(pool_size=(2,2))(con_layer)
drop_conv1 = Dropout(0.25)(pool_layer)

con_layer2 = Conv2D(64, kernel_size=(3,3), activation='relu', padding='same')(drop_conv1)
pool_layer2 = MaxPooling2D(pool_size=(2,2))(con_layer2)
drop_conv2 = Dropout(0.25)(pool_layer2)

con_layer3 = Conv2D(128, kernel_size=(3,3), activation='relu', padding='same')(drop_conv2)
pool_layer3 = MaxPooling2D(pool_size=(2,2))(con_layer3)
drop_conv3 = Dropout(0.25)(pool_layer3)

flat_G = Flatten()(drop_conv3)
hid_layer = Dense(256, activation='relu')(flat_G)
dropout = Dropout(0.5)(hid_layer)
hid_layer2 = Dense(128, activation='relu')(dropout)
dropout2 = Dropout(0.5)(hid_layer2)
out_layer = Dense(5, activation='softmax')(dropout2)

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import sparse_categorical_crossentropy
from tensorflow.keras.callbacks import EarlyStopping

model = Model([inpx], out_layer)
model.compile(optimizer=Adam(learning_rate=0.001), loss=sparse_categorical_crossentropy , metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [ ]:
model.fit(x_train, y_train, batch_size=32, epochs=30,callbacks=[early_stopping])

Epoch 1/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 23s 32ms/step - accuracy: 0.3039 - loss: 1.6635
Epoch 2/30
  9/392 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.3419 - loss: 1.5132

/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


392/392 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.3420 - loss: 1.5160
Epoch 3/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.3470 - loss: 1.5049
Epoch 4/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.3576 - loss: 1.4954
Epoch 5/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.3604 - loss: 1.4861
Epoch 6/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.3742 - loss: 1.4748
Epoch 7/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.3803 - loss: 1.4552
Epoch 8/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.4017 - loss: 1.4262
Epoch 9/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.4163 - loss: 1.3995
Epoch 10/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.4407 - loss: 1.3571
Epoch 11/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.4574 - loss: 1.3197
Epoch 12/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.4844 - loss: 1.2648
Epoch 13/30
392/392 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/st

In [ ]:
score = model.evaluate(x_test, y_test, verbose=0)
model.save("/content/drive/MyDrive/DS4420 project/cnn_model.keras")
print('loss=', score[0])
print('accuracy=', score[1])

loss= 1.988330364227295
accuracy= 0.4039565920829773


# A store model that predicts room types that are already filted down as a store.

In [ ]:
def folder_to_numpy_flat(folder_path, target_size=(128, 128)):
    images = []
    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)
        if not os.path.isfile(fpath):
            continue
        try:
            with Image.open(fpath) as img:
                img = img.convert('L')
                img = ImageOps.pad(img, target_size, color=0)
                images.append(np.array(img, dtype=np.uint8))
        except Exception as e:
            print(f"Skipped {fname}: {e}")
    return np.array(images, dtype=np.uint8)

In [ ]:
store_rooms = [
    "mall", "bookstore", "clothingstore", "videostore", "grocerystore",
    "jewelleryshop", "toystore", "shoeshop", "florist",
    "laundromat", "deli", "bakery",
]

store_room_to_label = {room: idx for idx, room in enumerate(store_rooms)}

store_dir = os.path.join(images_dir, 'store')
all_data = []
all_labels = []

# add labels for the room types
for room in os.listdir(store_dir):
    room_path = os.path.join(store_dir, room)
    if not os.path.isdir(room_path):
        continue
    label = store_room_to_label.get(room)
    if label is None:
        continue
    data = folder_to_numpy_flat(room_path)
    all_data.append(data)
    all_labels.append(np.full(data.shape[0], label))
    print(f"{room}: {label}")

store_data = np.concatenate(all_data, axis=0)
y_store = np.concatenate(all_labels, axis=0)
print(f"Total: {store_data.shape[0]} images, {len(store_rooms)} classes")

Loaded: toystore → label 6
Loaded: florist → label 8
Loaded: bakery → label 11
Loaded: laundromat → label 9
Loaded: bookstore → label 1
Loaded: grocerystore → label 4
Loaded: jewelleryshop → label 5
Loaded: shoeshop → label 7
Loaded: clothingstore → label 2
Loaded: videostore → label 3
Loaded: mall → label 0
Loaded: deli → label 10
Total: 2647 images, 12 classes


In [ ]:
x_train_s, x_test_s, y_train_s, y_test_s = train_test_split(
    store_data, y_store, test_size=0.2, random_state=1, stratify=y_store
)

x_train_s = x_train_s / 255.0
x_test_s  = x_test_s  / 255.0

x_train_s = x_train_s[..., np.newaxis]
x_test_s  = x_test_s[..., np.newaxis]

In [ ]:
inpx_s = Input(shape=(128, 128, 1))

con_layer = Conv2D(32, kernel_size=(3,3), activation='relu', padding='same')(inpx_s)
pool_layer = MaxPooling2D(pool_size=(2,2))(con_layer)
drop_conv1 = Dropout(0.25)(pool_layer)

con_layer2 = Conv2D(64, kernel_size=(3,3), activation='relu', padding='same')(drop_conv1)
pool_layer2 = MaxPooling2D(pool_size=(2,2))(con_layer2)
drop_conv2 = Dropout(0.25)(pool_layer2)

con_layer3 = Conv2D(128, kernel_size=(3,3), activation='relu', padding='same')(drop_conv2)
pool_layer3 = MaxPooling2D(pool_size=(2,2))(con_layer3)
drop_conv3 = Dropout(0.25)(pool_layer3)

flat_s = Flatten()(drop_conv3)
hid_layer_s = Dense(256, activation='relu')(flat_s)
dropout_s = Dropout(0.5)(hid_layer_s)
hid_layer2_s = Dense(128, activation='relu')(dropout_s)
dropout2_s = Dropout(0.5)(hid_layer2_s)
out_layer_s = Dense(len(store_rooms), activation='softmax')(dropout2_s)

In [ ]:
model_store = Model([inpx_s], out_layer_s)
model_store.compile(optimizer=Adam(learning_rate=0.001),
                    loss=sparse_categorical_crossentropy,
                    metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


In [ ]:
model_store.fit(x_train_s, y_train_s,
                batch_size=32,
                epochs=50,
                validation_split=0.1,
                callbacks=[early_stopping])

Epoch 1/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 19s 189ms/step - accuracy: 0.1318 - loss: 2.4406 - val_accuracy: 0.1698 - val_loss: 2.3919
Epoch 2/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.1344 - loss: 2.4000 - val_accuracy: 0.1698 - val_loss: 2.3983
Epoch 3/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.1549 - loss: 2.3762 - val_accuracy: 0.2170 - val_loss: 2.3582
Epoch 4/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.1554 - loss: 2.3533 - val_accuracy: 0.1981 - val_loss: 2.3385
Epoch 5/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.1585 - loss: 2.3477 - val_accuracy: 0.1887 - val_loss: 2.3383
Epoch 6/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.1706 - loss: 2.3224 - val_accuracy: 0.1934 - val_loss: 2.3142
Epoch 7/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.1822 - loss: 2.2948 - val_accuracy: 0.1887 - val_loss: 2.2977
Epoch 8/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.1927 - loss: 2.2827 - val_accuracy: 0.1698 -

In [ ]:
score_model = model_store.evaluate(x_test, y_test, verbose=0)
model.save("/content/drive/MyDrive/DS4420 project/store_cnn_model.keras")
print('loss=', score_model[0])
print('accuracy=', score_model[1])

loss= 3.159653663635254
accuracy= 0.06924058496952057
